# Recurrent Neural Network

x₁ → [RNN cell] → h₁ → y₁
       ↑
     h₀ (initial zero)

x₂ → [RNN cell] → h₂ → y₂
       ↑
     h₁ (from previous)

x₃ → [RNN cell] → h₃ → y₃

In [47]:
# Recurrent neural network.

import numpy as np

text = open('sample.txt','r',encoding='utf-8').read()
chars = sorted(list(set(text)))
data_size , vocab_size = len(text),len(chars)

print(f"Data has {data_size} character, {vocab_size} unique. ")

char_to_ix = {ch : i for i,ch in enumerate(chars)}
ix_to_char = {i:ch for i ,ch in enumerate(chars)}

print(f"Char to index {char_to_ix}   \n\n index to char {ix_to_char}")


Data has 2730 character, 44 unique. 
Char to index {'\n': 0, ' ': 1, '"': 2, ',': 3, '.': 4, ':': 5, '?': 6, 'A': 7, 'B': 8, 'E': 9, 'H': 10, 'I': 11, 'O': 12, 'S': 13, 'T': 14, 'W': 15, 'Y': 16, 'a': 17, 'b': 18, 'c': 19, 'd': 20, 'e': 21, 'f': 22, 'g': 23, 'h': 24, 'i': 25, 'j': 26, 'k': 27, 'l': 28, 'm': 29, 'n': 30, 'o': 31, 'p': 32, 'q': 33, 'r': 34, 's': 35, 't': 36, 'u': 37, 'v': 38, 'w': 39, 'x': 40, 'y': 41, '—': 42, '…': 43}   

 index to char {0: '\n', 1: ' ', 2: '"', 3: ',', 4: '.', 5: ':', 6: '?', 7: 'A', 8: 'B', 9: 'E', 10: 'H', 11: 'I', 12: 'O', 13: 'S', 14: 'T', 15: 'W', 16: 'Y', 17: 'a', 18: 'b', 19: 'c', 20: 'd', 21: 'e', 22: 'f', 23: 'g', 24: 'h', 25: 'i', 26: 'j', 27: 'k', 28: 'l', 29: 'm', 30: 'n', 31: 'o', 32: 'p', 33: 'q', 34: 'r', 35: 's', 36: 't', 37: 'u', 38: 'v', 39: 'w', 40: 'x', 41: 'y', 42: '—', 43: '…'}


In [46]:
# model parameters

hidden_size = 128
seq_length = 25
learning_rate = 1e-2

Wxh = np.random.randn(hidden_size,vocab_size) * 0.01
Whh = np.random.randn(hidden_size,hidden_size) * 0.01
Why = np.random.randn(vocab_size,hidden_size) * 0.01

bh = np.zeros((hidden_size,1))
by = np.zeros((vocab_size,1))


In [58]:
# Helper functions

def softmax(z):
    e_z = np.exp(z- np.max(z))
    return e_z / e_z.sum(axis=0)

def lossFun(inputs,targets, hidden_prev):
    xs,hs,ys,ps  = {},{},{},{}
    hs[-1] = np.copy(hidden_prev)
    loss = 0

    # forward pass
    for t in range(len(inputs)):
        xs[t] = np.zeros((vocab_size,1))
        xs[t][inputs[t]]=1
        hs[t] = np.tanh(np.dot(Wxh,xs[t]) + np.dot(Whh,hs[t-1]) +bh)
        ys[t] = np.dot(Why, hs[t]) + by
        ps[t] = softmax(ys[t])
        loss += -np.log(ps[t][targets[t],0])

    #BPTT

    dWxh,dWhh,dWhy = np.zeros_like(Wxh), np.zeros_like(Whh),np.zeros_like(Why)
    dbh, dby = np.zeros_like(bh), np.zeros_like(by)
    dhnext = np.zeros_like(hs[0])

    for t in reversed(range(len(inputs))):
        dy = np.copy(ps[t])
        dy[targets[t]]-=1
        dWhy+=np.dot(dy,hs[t].T)
        dby+=dy
        dh = np.dot(Why.T,dy) + dhnext
        dhraw = (1- hs[t]*hs[t]) *dh
        dbh+=dhraw
        dWxh+=np.dot(dhraw,xs[t].T)
        dWhh+=np.dot(dhraw,hs[t-1].T)
        dhnext=np.dot(Whh.T,dhraw)

    for dparam in [dWxh,dWhh,dWhy,dbh,dby]:
        np.clip(dparam,-5,5,out=dparam)
    return loss,dWxh,dWhh,dWhy,dbh,dby,hs[len(inputs)-1]



